# DS event-sync benchmark

Mirrors the integration test `test_file_deletion_do_to_ds` (real GDrive, single DO, single DS) and just times the parts we care about. Run cells top-to-bottom. If anything fails, run the smoke cell on its own first to isolate.

In [ ]:
import os
import time
from pathlib import Path

os.environ['PRE_SYNC'] = 'false'

from syft_client.sync.syftbox_manager import SyftboxManager

# --- config ---
CREDENTIALS_DIR = Path('~/Desktop/beach-testing').expanduser()
EMAIL_DO = 'beach.do.007@gmail.com'
EMAIL_DS = 'beach.ds.008@gmail.com'
TOKEN_DO = CREDENTIALS_DIR / 'token-do-1.json'
TOKEN_DS = CREDENTIALS_DIR / 'token-ds-2.json'

NUM_EVENTS = 100
EVENT_BYTES = 256

## 1. Wipe leftover state
Same helper the integration test fixture uses (`remove_syftboxes_from_drive`).

In [ ]:
_ds, _do = SyftboxManager._pair_with_google_drive_testing_connection(
    do_email=EMAIL_DO, ds_email=EMAIL_DS,
    do_token_path=TOKEN_DO, ds_token_path=TOKEN_DS,
    add_peers=False,
)
_ds.delete_syftbox(broadcast_delete_events=False)
_do.delete_syftbox(broadcast_delete_events=False)
del _ds, _do
print('cleanup done')

## 2. Smoke test — one file end-to-end
Literal copy of the integration test path: pair → grant read → DO writes → `do.sync()` → `ds.sync()`. If this fails, the bigger benchmark won't work either, so we stop here and debug.

In [ ]:
ds, do = SyftboxManager._pair_with_google_drive_testing_connection(
    do_email=EMAIL_DO, ds_email=EMAIL_DS,
    do_token_path=TOKEN_DO, ds_token_path=TOKEN_DS,
    add_peers=True,
    use_in_memory_cache=False,
)
do.datasite_owner_syncer.perm_context.open('.').grant_read_access(ds.email)

smoke_path = do.syftbox_folder / do.email / 'smoke.txt'
smoke_path.parent.mkdir(parents=True, exist_ok=True)
smoke_path.write_text('hello')

do.sync()
ds.sync()

smoke_received = (ds.syftbox_folder / do.email / 'smoke.txt').exists()
print(f'smoke received on DS: {smoke_received}')
assert smoke_received, 'smoke test failed — basic event flow is broken, do not run benchmark'

## 3. Publish N events on DO
DO writes N files to its datasite, syncing once at the end (each file is its own `FileChangeEvent`).

In [ ]:
do_dir = do.syftbox_folder / do.email
body = 'x' * (EVENT_BYTES - 1) + '\n'

t0 = time.time()
for i in range(NUM_EVENTS):
    (do_dir / f'event_{i}.txt').write_text(body)
    do.sync(auto_checkpoint=False)
publish_time = time.time() - t0
event_count = len(do._get_all_accepted_events_do())
print(f'DO publish (write+sync): {publish_time:.2f}s, {event_count} events on owner log')

In [ ]:
event_count = len(do._get_all_accepted_events_do())
event_count

In [ ]:
do.compact_outboxes()

## 4. KEY METRIC — DS pulls events down

In [ ]:
t0 = time.time()
ds.sync()
ds_sync_time = time.time() - t0
files_seen = len(list((ds.syftbox_folder / do.email).rglob('event_*.txt')))
per_event = ds_sync_time / max(event_count, 1)
print(f'DS sync: {ds_sync_time:.2f}s   ({per_event:.4f}s/event)')
print(f'DS sees {files_seen}/{event_count} event files')

In [ ]:
ds.syftbox_folder

In [ ]:
ds.syftbox_folder

In [ ]:
# Second sync — warm cache.
t0 = time.time()
ds.sync()
print(f'DS second sync: {time.time() - t0:.2f}s')